# 端侧现场交付状态自动说明助手

> Intel AI PC 创新应用征文 & OpenClaw Skill 挑战赛作品 Notebook

本项目面向企业园区、医院科室、工厂仓库等弱网、内网和隐私敏感场景。
现场人员只需要拍一张交付照片，并提供少量必要信息，系统即可在端侧生成：

- VLM 图片理解结果
- 收件人送达通知
- TTS 播报音频
- ASR 回读验证或语音追问入口
- Markdown 交付留档

整体流程如下：

```text
交付现场照片
  -> OpenVINO Qwen3-VL 图片理解
  -> 结构化交付信息
  -> 场景化通知生成
  -> OpenVINO Qwen3-TTS 语音播报
  -> OpenVINO Qwen3-ASR 语音回读/追问
  -> Markdown / JSON 留档
```


## 1. 安装依赖

本 Notebook 对齐官方 `modelscope-workshop` baseline。

修复重点：如果加载 VLM 时出现 `KeyError: 'qwen3_vl'`，通常说明当前环境的
`transformers / optimum-intel` 不支持 Qwen3-VL。请重新运行本单元，安装官方固定版本依赖。


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt --upgrade


## 2. 导入模块与读取样例

`samples/manifest.example.json` 只包含必要上下文，不预置最终回复。
最终通知、TTS 文本和追问回答均由模型和代码实时生成。


In [ ]:
from pathlib import Path
import json
import time

import IPython.display as ipd
from IPython.display import Image, Markdown, display

from src.delivery_generator import generate_delivery_result
from src.delivery_schema import VisionObservation
from src.openvino_vlm_service import (
    DEFAULT_VLM_MODEL_DIR,
    DEFAULT_VLM_MODEL_ID,
    OpenVINODeliveryVLM,
    download_vlm_model,
)
from src.sample_loader import context_from_sample, load_manifest
from src.workshop_services import load_asr_model, load_tts_model

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

samples = load_manifest("samples/manifest.example.json")
print(f"已加载 {len(samples)} 个交付场景样例")
for sample in samples:
    print(f"- {sample['id']}: {sample['scene']} -> {sample['image_path']}")


## 3. 展示三类交付场景

三张图片分别代表企业、医院、工厂三类交付现场。
这里展示的是输入素材，后续输出由模型实时生成。


In [ ]:
for sample in samples:
    print("\n" + "=" * 80)
    print(f"样例：{sample['id']}")
    print(sample["image_description"])
    display(Image(filename=sample["image_path"], width=430))


## 4. 下载并加载 OpenVINO VLM

默认模型：`snake7gun/Qwen3-VL-4B-Instruct-int4-ov`。

加载方式参考官方 Lab 1：

```python
from optimum.intel.openvino import OVModelForVisualCausalLM
model = OVModelForVisualCausalLM.from_pretrained(model_dir, device=device)
processor.apply_chat_template(..., tokenize=True, return_dict=True, return_tensors="pt")
```

如果这里仍出现 `qwen3_vl` 相关错误，请确认上一节依赖安装成功后重启 Kernel 再运行。


In [ ]:
DEVICE_VLM = "AUTO"

start = time.time()
model_dir = download_vlm_model(DEFAULT_VLM_MODEL_ID, DEFAULT_VLM_MODEL_DIR)
print("模型目录:", model_dir)

vlm_service = OpenVINODeliveryVLM(model_dir=model_dir, device=DEVICE_VLM)
print(f"OpenVINO VLM 加载完成，用时 {time.time() - start:.1f}s")


## 5. VLM 实时推理并生成送达说明

每张图片会先由 VLM 输出结构化观察，再进入场景化通知生成逻辑。


In [ ]:
outputs = []

for sample in samples:
    print("\n" + "=" * 90)
    print(f"处理样例：{sample['id']}")
    display(Image(filename=sample["image_path"], width=430))

    context = context_from_sample(sample)
    followup_question = ""
    if sample.get("followup_cases"):
        followup_question = sample["followup_cases"][0].get("question", "")

    observation = vlm_service.observe_delivery_image(sample["image_path"])
    result = generate_delivery_result(context, observation, followup_question)

    record = {
        "sample_id": sample["id"],
        "image_path": sample["image_path"],
        "context": context.__dict__,
        "observation": observation.__dict__,
        "result": result.to_dict(),
    }
    outputs.append(record)

    print("【VLM 结构化观察】")
    print(json.dumps(observation.__dict__, ensure_ascii=False, indent=2))
    print("\n【收件人通知】")
    print(result.recipient_message)
    print("\n【TTS 播报文本】")
    print(result.tts_text)
    if result.followup_answer:
        print("\n【二次追问】")
        print("问：", followup_question)
        print("答：", result.followup_answer)

    (OUTPUT_DIR / f"{sample['id']}_record.md").write_text(result.archive_markdown, encoding="utf-8")

(OUTPUT_DIR / "delivery_results.json").write_text(
    json.dumps(outputs, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\n结构化结果已保存到 outputs/delivery_results.json")


## 6. OpenVINO TTS：生成第一条通知的语音

这里使用官方 Lab 3 的 `Qwen3-TTS-CustomVoice-0.6B-fp16-ov` helper。

为了控制运行时间，默认只对第一条交付通知生成音频。


In [ ]:
RUN_TTS = True
DEVICE_AUDIO = "CPU"

tts_audio_path = OUTPUT_DIR / "office_001_tts.wav"

if RUN_TTS:
    start = time.time()
    tts_model = load_tts_model(device=DEVICE_AUDIO)
    print(f"TTS 模型加载完成，用时 {time.time() - start:.1f}s")

    text = outputs[0]["result"]["tts_text"]
    print("TTS 输入文本：")
    print(text)

    start = time.time()
    wavs, sr = tts_model.generate_custom_voice(
        text=text,
        speaker="vivian",
        language="Chinese",
        instruct="用自然、清晰、礼貌的语气播报现场交付通知。",
        non_streaming_mode=True,
        max_new_tokens=2048,
    )
    tts_time = time.time() - start

    if wavs is not None:
        import soundfile as sf
        sf.write(tts_audio_path, wavs[0], sr)
        duration = len(wavs[0]) / sr
        print(f"语音已生成：{tts_audio_path}")
        print(f"音频时长: {duration:.2f}s，推理耗时: {tts_time:.2f}s，RTF: {tts_time / duration:.3f}")
        ipd.display(ipd.Audio(wavs[0], rate=sr))


## 7. OpenVINO ASR：回读 TTS 音频或识别语音追问

ASR 有两个用途：

1. 回读上一节生成的 TTS 音频，验证语音链路可用。
2. 后续接入配送员或收件人的语音追问，例如“我还是找不到，旁边有什么标志？”。

这里使用官方 Lab 2 的 `Qwen3-ASR-0.6B-fp16-ov` helper。


In [ ]:
RUN_ASR = True

if RUN_ASR:
    start = time.time()
    asr_model = load_asr_model(device=DEVICE_AUDIO)
    print(f"ASR 模型加载完成，用时 {time.time() - start:.1f}s")

    audio_for_asr = tts_audio_path if tts_audio_path.exists() else None
    if audio_for_asr is None:
        print("未找到 TTS 音频，跳过 ASR 回读。")
    else:
        results = asr_model.transcribe(audio=str(audio_for_asr), language=None)
        print("ASR 回读结果：")
        print("语言：", results[0].language)
        print("文本：", results[0].text)


## 8. 交付记录展示

每个样例都会导出一份 Markdown 留档，可用于企业内网、医院交接记录或工厂工单记录。


In [ ]:
for path in sorted(OUTPUT_DIR.glob("*_record.md")):
    print("\n" + "-" * 80)
    print(path)
    display(Markdown(path.read_text(encoding="utf-8")))


## 9. 可选：启动 Gradio 演示

如果灵感流环境允许端口访问，可以启动一个简易 Web 演示界面。


In [ ]:
# from src.gradio_service import build_demo
# demo = build_demo()
# demo.launch(server_name="0.0.0.0", server_port=7860, share=False)


## 10. 小结

本 Notebook 已形成完整比赛展示闭环：

- 三类隐私敏感交付场景
- OpenVINO VLM 实时图片理解
- 场景化送达说明生成
- OpenVINO TTS 语音播报
- OpenVINO ASR 回读/语音追问入口
- Markdown 与 JSON 留档

后续在研习社文章中补充本 Notebook 的运行截图、GitHub 仓库地址和 Copaw Skill 运行截图即可。
